In [4]:
import data_generation_utils as dgu

instances = dgu.read_instances("instances.txt")
instances

[Instance(n_processors=4, tasks=[Task(id=0, r=0, l=1, w=10), Task(id=1, r=2, l=6, w=2), Task(id=2, r=6, l=8, w=12), Task(id=3, r=2, l=2, w=8), Task(id=4, r=5, l=6, w=6), Task(id=5, r=3, l=6, w=7), Task(id=6, r=1, l=7, w=2), Task(id=7, r=5, l=8, w=12), Task(id=8, r=6, l=6, w=3), Task(id=9, r=5, l=10, w=12), Task(id=10, r=1, l=10, w=7), Task(id=11, r=5, l=12, w=5)])]

In [5]:
import numpy as np
from math import factorial
from itertools import permutations
from data_generation_utils import Instance, Task


def evaluate_schedule(order, n_processors):
    """
    Schedule tasks in the given order on identical processors.

    Each task is assigned to the processor that becomes available
    earliest. Release times are respected.

    Returns:
        weighted_completion_time
        schedule
    """

    # When each processor becomes available
    processor_available = [0] * n_processors

    # (task_id, processor, start, completion)
    schedule = []

    objective = 0

    for task in order:
        # Find processor that becomes available first
        processor = min(
            range(n_processors),
            key=lambda p: processor_available[p]
        )

        start = max(
            processor_available[processor],
            task.r
        )

        completion = start + task.l

        processor_available[processor] = completion

        objective += task.w * completion

        schedule.append(
            (task.id, processor, start, completion)
        )

    return objective, schedule


def brute_force(instance: Instance):
    """
    Find the optimal schedule by enumerating all task permutations.

    WARNING:
        Complexity is O(n! * n * m), so this is only practical
        for small instances.
    """

    best_objective = float("inf")
    best_order = None
    best_schedule = None

    i = 0
    from math import factorial
    total = factorial(len(instance.tasks))
    for order in permutations(instance.tasks):

        objective, schedule = evaluate_schedule(
            order,
            instance.n_processors
        )

        if objective < best_objective:
            best_objective = objective
            best_order = order
            best_schedule = schedule
            
                # Print every 1% of progress
        if i % max(1, total // 100) == 0 or i == total:
            progress = i / total * 100

            print(
                f"\rProgress: {progress:6.2f}% "
                f"({i:,}/{total:,}) "
                f"Best objective: {best_objective:,}",
                end=""
            )
        i+=1

    return best_objective, best_order, best_schedule


    


In [6]:
result = brute_force(instances[0])
result

Progress:  99.00% (474,211,584/479,001,600) Best objective: 1,106

(1106,
 (Task(id=0, r=0, l=1, w=10),
  Task(id=3, r=2, l=2, w=8),
  Task(id=5, r=3, l=6, w=7),
  Task(id=7, r=5, l=8, w=12),
  Task(id=10, r=1, l=10, w=7),
  Task(id=9, r=5, l=10, w=12),
  Task(id=2, r=6, l=8, w=12),
  Task(id=4, r=5, l=6, w=6),
  Task(id=11, r=5, l=12, w=5),
  Task(id=8, r=6, l=6, w=3),
  Task(id=1, r=2, l=6, w=2),
  Task(id=6, r=1, l=7, w=2)),
 [(0, 0, 0, 1),
  (3, 1, 2, 4),
  (5, 2, 3, 9),
  (7, 3, 5, 13),
  (10, 0, 1, 11),
  (9, 1, 5, 15),
  (2, 2, 9, 17),
  (4, 0, 11, 17),
  (11, 3, 13, 25),
  (8, 1, 15, 21),
  (1, 0, 17, 23),
  (6, 2, 17, 24)])